# 00 · Project quickstart and architecture

Run every cell without downloading data or models. This verifies the real Python inference path using an explicitly synthetic sample catalog. It does **not** train or benchmark a model.

Install from the project directory: `python -m pip install -e ".[notebooks,dev]"`. Start the UI with `python -m uvicorn cinescope.api:app --host 127.0.0.1 --port 8000` and open `http://127.0.0.1:8000`.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from CineScope/notebooks or CineScope.'
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)
(ROOT / 'artifacts').mkdir(exist_ok=True)
SEED = 42

## Architecture
MovieLens ratings/tags + linked IMDb/TMDB metadata → canonical IDs and Parquet → Spark ETL and ALS → saved factors.

IMDb sentiment text → leakage checks → TF-IDF/DistilBERT → saved classifier.

Historical/live reviews → the same analyzer → sentence-level aspect evidence + review screening → full-population aggregates and extractive summary.

Ratings + content + optional ALS → hybrid ranking → grounded conversational retrieval → frontend. Browser speech or local Whisper produces text before retrieval. The optional local transformer rewrites the retrieved facts.

The HTTP path is synchronous for interactive use. Spark Structured Streaming processes durable exported events for analytics; it is not on the interactive critical path.

In [2]:
from cinescope.reviews import ReviewAnalyzer
analyzer = ReviewAnalyzer(mode='lexicon')
result = analyzer.analyze('The story is clever and the acting is excellent, but the pacing is slow. The visuals are stunning.')
result

{'text': 'The story is clever and the acting is excellent, but the pacing is slow. The visuals are stunning.', 'sentiment': {'label': 'positive', 'sentiment_index': 84.3, 'positive_probability': None, 'confidence': None, 'uncertain': False, 'method': 'lexicon-baseline'}, 'aspects': {'Story': {'score': 90.0, 'mentions': 1, 'scored_mentions': 1, 'evidence': ['The story is clever and the acting is excellent'], 'method': 'clause-mention + lexicon-baseline'}, 'Acting': {'score': 90.0, 'mentions': 1, 'scored_mentions': 1, 'evidence': ['The story is clever and the acting is excellent'], 'method': 'clause-mention + lexicon-baseline'}, 'Visuals': {'score': 84.3, 'mentions': 1, 'scored_mentions': 1, 'evidence': ['The visuals are stunning'], 'method': 'clause-mention + lexicon-baseline'}, 'Music': {'score': None, 'mentions': 0, 'scored_mentions': 0, 'evidence': [], 'method': 'clause-mention + lexicon-baseline'}, 'Pacing': {'score': 15.7, 'mentions': 1, 'scored_mentions': 1, 'evidence': ['the paci

In [3]:
assert result['aspects']['Acting']['score'] > 50
assert result['aspects']['Pacing']['score'] < 50
assert result['aspects']['Ending']['score'] is None
print('Mixed sentiment and missing evidence handled correctly.')

Mixed sentiment and missing evidence handled correctly.


In [4]:
import tempfile
from cinescope.api import create_app
from fastapi.testclient import TestClient
with tempfile.TemporaryDirectory() as directory:
    client = TestClient(create_app(data_dir=directory, sentiment_mode='lexicon'))
    print(client.get('/api/health').json())
    print(client.get('/api/movies/1').json()['review_summary']['summary'])
    print(client.get('/api/movies/13').json()['review_summary']['summary'])
    client.put('/api/users/student/ratings', json={'movie_id':3,'rating':5})
    picks = client.get('/api/users/student/recommendations', params={'genres':'Sci-Fi,Romance'}).json()['movies']
    print([(m['title'],m['reasons']) for m in picks])
    assert all({'Sci-Fi','Romance'}.issubset(m['genres']) for m in picks)
    assert all(m['movie_id'] != 3 for m in picks)

/root/.local/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
{'status': 'ok', 'catalog_size': 13, 'demo_data': True, 'sentiment_model': 'lexicon-baseline', 'als_loaded': False, 'semantic_retrieval': False, 'chat_transformer': False, 'speech_enabled': False}
Across 3 eligible reviews, 3 lean positive and 0 lean negative; 0 are uncertain. Praise centers on visuals, acting, music. Criticism centers on pacing. This is a small sample.
No eligible reviews yet. Recommendations can still use the movie’s genres, themes and synopsis.
[('Eternal Sunshine of the Spotless Mind', ['Matches your highly rated Sci-Fi + Romance + Drama films']), ('The Last Signal', ['Matches your highly rated Sci-Fi + Romance + Love films'])]


## Run order
01 prepares data. 02 trains the CPU baseline. 03 optionally fine-tunes transformers. 04 audits reviews and creates topics/summaries. 05 trains distributed recommendations. 06 adds semantic discovery/speech. 07 evaluates saved outputs. 08 covers additional linguistics/sequence/graph concepts.

Restart the API after creating new model artifacts. `/api/health` reports exactly what was loaded. See `docs/SYLLABUS_MAP.md`, `docs/ARCHITECTURE.md`, and `docs/DATASETS.md`.